### Imports

In [ ]:
import numpy as np
import pandas as pd
from pyomo.environ import *
from bluebikes_rebalancing.config import LOCAL_DATA_DIR
from bluebikes_rebalancing.plots import plot_rebalancing_map
from bluebikes_rebalancing.model import build_vrp_model

### Parameters

In [ ]:
# Model parameters
Q = 20              # truck capacity (bikes)
K = 1               # single vehicle (multi-vehicle model with K=1 ≈ single-vehicle)
S = Q               # depot capacity = truck capacity to mimic single-vehicle sourcing
GAMMA = 0.0         # no deployment cost
B = 2               # buffer (min bikes and docks at each station)
ALPHA = 1.0         # distance cost weight (fixed)
BETA = 10           # service quality weight (selected from parametric analysis)
SERVICE_TIME = 5.0  # minutes per station stop
TIME_PER_BIKE = 1.0 # minutes per bike loaded/unloaded
T_MAX = 180.0       # operational window (minutes)

# Date to optimize
DATE = "20260302"   # available: 20260302–20260306

depot_names = ["depot_start", "depot_end"]

### Load Data

In [3]:
# Data directories
stations_dir = LOCAL_DATA_DIR / "processed" / "stations"
initial_status_dir = LOCAL_DATA_DIR / "processed" / "initial_status"
demand_dir = LOCAL_DATA_DIR / "processed" / "demand"
network_dir = LOCAL_DATA_DIR / "processed" / "network"

In [4]:
# Station metadata and road network (date-independent)
stations_df = pd.read_csv(stations_dir / "station_information.csv", index_col="idx")

network_df = pd.read_csv(network_dir / "dist_ttime_long.csv")
network_df["ttime_min"] = network_df["ttime_s"] / 60
idx_to_name = stations_df["short_name"].to_dict()
network_df["origin"] = network_df["origin_idx"].map(idx_to_name)
network_df["dest"] = network_df["dest_idx"].map(idx_to_name)

In [33]:
network_df

,origin_idx,dest_idx,dist_m,ttime_s,ttime_min,origin,dest
0,0,1,1960.410788,240.718181,4.011970,depot_start,A32003
1,0,2,0.000000,0.000000,0.000000,depot_start,A32004
2,0,3,1429.121279,163.941600,2.732360,depot_start,A32008
3,0,4,462.312933,75.212156,1.253536,depot_start,A32056
4,0,5,2696.517676,350.551370,5.842523,depot_start,B32000
...,...,...,...,...,...,...,...
2347,48,43,2712.058013,344.698726,5.744979,depot_end,D32032
2348,48,44,2420.634004,345.802329,5.763372,depot_end,D32061
2349,48,45,1503.753237,179.808072,2.996801,depot_end,K32007
2350,48,46,1043.823684,125.038412,2.083974,depot_end,K32015


In [5]:
# Date-specific data: initial bike counts and morning demand forecasts
initial_status_df = pd.read_csv(
    initial_status_dir / f"initial_status_{DATE}.csv", index_col="idx"
)
demand_df = pd.read_csv(demand_dir / f"demand_{DATE}.csv", index_col="idx")

### Filter to active stations

Stations missing from the daily snapshot are excluded from the optimization entirely.

In [6]:
active_stations = initial_status_df["short_name"].tolist()
active_stations_with_depots = active_stations + depot_names

sdf = stations_df[stations_df["short_name"].isin(active_stations_with_depots)].copy()
ddf = demand_df[demand_df["station_id"].isin(active_stations)].copy()
idf = initial_status_df[initial_status_df["short_name"].isin(active_stations)].copy()
ndf = network_df[
    network_df["origin"].isin(active_stations_with_depots)
    & network_df["dest"].isin(active_stations_with_depots)
].copy()

print(f"{len(active_stations)} active stations, {len(ndf)} arcs")

44 active stations, 2070 arcs


### Prepare Model Inputs

In [7]:
nodes = sdf["short_name"].tolist()
stations = [n for n in nodes if n not in depot_names]

# Initial inventory — depots get special values
b = idf.set_index("short_name")["initial_status"].to_dict()
b["depot_start"] = Q
b["depot_end"] = 0

# Docking capacity — both depots set to Q
c = sdf.set_index("short_name")["capacity"].to_dict()
c["depot_start"] = Q
c["depot_end"] = Q

# Net demand and target inventory per station
ddf["d"] = ddf["dropoffs_forecast"] - ddf["pickups_forecast"]
d = ddf.set_index("station_id")["d"].to_dict()
t = {i: c[i] / 2 - d[i] for i in stations}

# Distance and travel time matrices keyed by (origin, destination) short_name pairs
dist = {(row.origin, row.dest): row.dist_m for row in ndf.itertuples()}
ttime = {(row.origin, row.dest): row.ttime_min for row in ndf.itertuples()}

In [8]:
deviations = [b[i] - t[i] for i in stations]
print(f"Average target inventory per station: {np.array(list(t.values())).mean():.1f} bikes")
print(f"Initial MAE from target: {np.mean([abs(d) for d in deviations]):.1f} bikes")
print(f"Initial squared deviation: {np.sum([d**2 for d in deviations]):.1f} bikes²")

Average target inventory per station: 8.1 bikes
Initial MAE from target: 5.1 bikes
Initial squared deviation: 1631.5 bikes²


### Build Model

In [ ]:
model = build_vrp_model(
    nodes=nodes,
    stations=stations,
    b=b,
    c=c,
    t=t,
    dist=dist,
    ttime=ttime,
    Q=Q,
    K=K,
    S=S,
    B=B,
    T_MAX=T_MAX,
    ALPHA=ALPHA,
    BETA=BETA,
    GAMMA=GAMMA,
    SERVICE_TIME=SERVICE_TIME,
    TIME_PER_BIKE=TIME_PER_BIKE,
)

### Solve

In [ ]:
solver = SolverFactory("gurobi")
solver.options["TimeLimit"] = 1500
solver.options["MIPGap"] = 0.02
solver.options["Threads"] = 4
result = solver.solve(model, tee=False)

status = str(result.solver.termination_condition)
total_distance_m = sum(
    value(model.dist[i, j]) * value(model.x[i, j, k])
    for (i, j) in model.ARCS for k in model.VEHICLES
)
total_time_m = sum(
    value(model.ttime[i, j]) * value(model.x[i, j, k])
    for (i, j) in model.ARCS for k in model.VEHICLES
)
deviations = [value(model.b_final[i]) - value(model.t[i]) for i in stations]
total_deviation_sq = sum(dev**2 for dev in deviations)
mean_abs_deviation = sum(abs(dev) for dev in deviations) / len(deviations)

In [ ]:
travel_time = sum(
    value(model.ttime[i, j]) * value(model.x[i, j, k])
    for (i, j) in model.ARCS for k in model.VEHICLES
)

station_time = sum(
    SERVICE_TIME * sum(value(model.x[i, j, k]) for (i, j) in model.ARCS if i == s for k in model.VEHICLES) +
    TIME_PER_BIKE * sum(value(model.u[s, k]) + value(model.v[s, k]) for k in model.VEHICLES)
    for s in stations
)

depot_time = TIME_PER_BIKE * sum(
    value(model.u["depot_start", k]) + value(model.v["depot_end", k]) for k in model.VEHICLES
)

total_time = travel_time + station_time + depot_time
print(f"Total operation time: {total_time:.1f} min (budget: {T_MAX:.0f} min)")

### Plot

In [ ]:
# --- Route ---
route_records = [
    {"from": i, "to": j, "x": sum(value(model.x[i, j, k]) for k in model.VEHICLES)}
    for (i, j) in model.ARCS
    if sum(value(model.x[i, j, k]) for k in model.VEHICLES) > 0.5
]
route_df = pd.DataFrame(route_records)

# --- Station operations (aggregated over the fleet) ---
ops_records = []
for i in model.NODES:
    pickups = sum(value(model.u[i, k]) for k in model.VEHICLES)
    dropoffs = sum(value(model.v[i, k]) for k in model.VEHICLES)
    load_leaving = sum(value(model.w[i, k]) for k in model.VEHICLES)
    if i in stations:
        final = value(model.b_final[i])
        target = value(model.t[i])
    elif i == "depot_start":
        final = value(model.b[i]) - pickups
        target = None
    else:  # depot_end
        final = value(model.b[i]) + dropoffs
        target = None
    ops_records.append(
        {
            "station": i,
            "initial": value(model.b[i]),
            "pickups": pickups,
            "dropoffs": dropoffs,
            "final": final,
            "target": target,
            "capacity": value(model.c[i]),
            "load_leaving": load_leaving,
        }
    )
ops_df = pd.DataFrame(ops_records)

print(f"Objective: {value(model.obj):.2f}")
print(f"\nRoute ({len(route_df)} arcs):")
print(route_df.to_string(index=False))
print(f"\nStation operations:")
print(ops_df.to_string(index=False))

In [ ]:
import geopandas as gpd

# --- Load road network routes ---
network_gdf = gpd.read_file(LOCAL_DATA_DIR / "processed" / "network" / "routes_long_wgs84" / "routes_long_wgs84.shp")

# Convert indices to station names
idx_to_name = stations_df["short_name"].to_dict()
network_gdf["origin"] = network_gdf["origin_idx"].map(idx_to_name)
network_gdf["dest"] = network_gdf["dest_idx"].map(idx_to_name)

In [ ]:
# --- Build van journey from solution (vehicle column for the fleet plot) ---
journey_records = [
    {"vehicle": k, "from": i, "to": j}
    for (i, j) in model.ARCS for k in model.VEHICLES
    if value(model.x[i, j, k]) > 0.5
]
journey_df = pd.DataFrame(journey_records)
journey_gdf = journey_df.merge(
    network_gdf[["origin", "dest", "geometry"]],
    left_on=["from", "to"], right_on=["origin", "dest"], how="left",
).drop(columns=["origin", "dest"])
journey_gdf = gpd.GeoDataFrame(journey_gdf, geometry="geometry", crs="EPSG:4326")

# --- Build stations plot dataframe ---
plot_df = ops_df[ops_df["station"].isin(stations)].copy()
plot_df = plot_df.merge(
    stations_df[["short_name", "lat", "lon"]],
    left_on="station", right_on="short_name"
).copy()

# Deviation from target (initial - target): negative = needs bikes, positive = has excess
plot_df["deviation"] = plot_df["initial"].astype(float) - plot_df["target"].astype(float)

In [15]:
plot_df

,station,initial,pickups,dropoffs,final,target,capacity,load_leaving,short_name,lat,lon,deviation
0,A32003,4,-0.0,-0.0,4.0,-0.5,11.0,2.0,A32003,42.350406,-71.108279,4.5
1,A32004,3,0.0,-0.0,3.0,-2.5,15.0,-0.0,A32004,42.338629,-71.106500,5.5
2,A32008,14,2.0,-0.0,12.0,8.5,15.0,7.0,A32008,42.347241,-71.105301,5.5
3,A32056,1,-0.0,1.0,2.0,1.5,25.0,4.0,A32056,42.336610,-71.110047,-0.5
4,B32001,11,-0.0,-0.0,11.0,10.5,21.0,2.0,B32001,42.339200,-71.096625,0.5
5,B32002,3,-0.0,-0.0,3.0,8.5,27.0,2.0,B32002,42.336244,-71.087986,-5.5
6,B32003,1,-0.0,1.0,2.0,-4.5,25.0,8.0,B32003,42.337417,-71.102861,5.5
7,B32005,34,9.0,-0.0,25.0,21.5,39.0,12.0,B32005,42.343666,-71.085824,12.5
8,B32006,14,9.0,-0.0,5.0,1.5,15.0,9.0,B32006,42.340115,-71.100619,12.5
9,B32009,15,0.0,-0.0,15.0,10.5,19.0,-0.0,B32009,42.341022,-71.092105,4.5


In [ ]:
depot_df = stations_df[stations_df["short_name"] == "depot_start"][["lat", "lon"]]

# Per-vehicle depot operations (load at depot_start, return at depot_end)
depot_ops = []
for k in model.VEHICLES:
    load = int(round(value(model.u["depot_start", k])))
    ret = int(round(value(model.v["depot_end", k])))
    if load > 0 or ret > 0:
        depot_ops.append({"vehicle": k, "pickups": load, "dropoffs": ret})

date_string = DATE[4:6] + "/" + DATE[6:] + "/" + DATE[:4]

plot_rebalancing_map(
    plot_df=plot_df,
    title=f"{date_string} Fleet Rebalancing Journey. (Beta = {BETA})",
    network_gdf=network_gdf,
    journey_gdf=journey_gdf,
    depot_df=depot_df,
    depot_ops=depot_ops,
)

In [17]:
try:
    obj_bound = float(result.problem.lower_bound)
    obj_value = float(result.problem.upper_bound)
    gap = abs(obj_value - obj_bound) / abs(obj_value)
    gap_str = f"{gap * 100:.2f}%"
except (AttributeError, TypeError, ZeroDivisionError):
    gap_str = "N/A"

print(f"Solver status:               {status}")
print(f"MIP gap:                     {gap_str}")
print(f"Total distance traveled:     {total_distance_m / 1000:.2f} km")
print(f"Total squared deviation:     {total_deviation_sq:.1f} bikes²")
print(f"Total operation time:        {total_time:.1f} min (budget: {T_MAX:.0f} min)")
print(f"  — travel:                  {travel_time:.1f} min ({travel_time / total_time * 100:.0f}%)")
print(f"  — station ops:             {station_time:.1f} min ({station_time / total_time * 100:.0f}%)")
print(f"  — depot handling:          {depot_time:.1f} min ({depot_time / total_time * 100:.0f}%)")

Solver status:               optimal
MIP gap:                     1.92%
Total distance traveled:     7.87 km
Total squared deviation:     745.5 bikes²
Total operation time:        178.8 min (budget: 180 min)
  — travel:                  18.8 min (10%)
  — station ops:             147.0 min (82%)
  — depot handling:          13.0 min (7%)


---

Create outputs

In [18]:
import json
from pathlib import Path
import pandas as pd
import geopandas as gpd
from pyomo.environ import value

In [19]:

# ============================================================================
# Helper function for route ordering
# ============================================================================

def order_route_by_sequence(route_df):
    """Order route dataframe by visit sequence starting from depot_start."""
    ordered = []
    current = "depot_start"
    
    while len(ordered) < len(route_df):
        # Find arc departing from current node
        arc = route_df[route_df["from"] == current]
        if arc.empty:
            break
        ordered.append(arc.iloc[0])
        current = arc.iloc[0]["to"]
    
    return pd.DataFrame(ordered).reset_index(drop=True)

In [ ]:
# ============================================================================
# 1. Parameters JSON
# ============================================================================

parameters = {
    "target_date": DATE,
    "truck_capacity": Q,
    "fleet_size": K,
    "depot_capacity": S,
    "buffer": B,
    "alpha": ALPHA,
    "beta": BETA,
    "gamma": GAMMA,
    "service_time": SERVICE_TIME,
    "time_per_bike": TIME_PER_BIKE,
    "max_operation_time": T_MAX,
    "solver": "gurobi",
    "solver_time_limit": 1500,
    "solver_mip_gap": 0.02,
    "solver_threads": 4,
    "num_stations": len(stations),
    "num_nodes": len(nodes),
    "num_vehicles": K,
    "M_low": {i: value(model.M_low[i]) for i in model.NODES},
    "M_up": {i: value(model.M_up[i]) for i in model.NODES},
}

# output_dir = Path("data/rebalancing_results") / DATE
# output_dir.mkdir(parents=True, exist_ok=True)

# with open(output_dir / "parameters.json", "w") as f:
#     json.dump(parameters, f, indent=2)

In [21]:
parameters

{'target_date': '20260302',
 'truck_capacity': 20,
 'buffer': 2,
 'alpha': 1.0,
 'beta': 10,
 'service_time': 5.0,
 'time_per_bike': 1.0,
 'max_operation_time': 180.0,
 'solver': 'gurobi',
 'solver_time_limit': 1500,
 'solver_mip_gap': 0.02,
 'solver_threads': 4,
 'num_stations': 44,
 'num_nodes': 46,
 'big_M': 54}

In [ ]:
# ============================================================================
# 2. Results Metrics JSON
# ============================================================================

# Extract objective components
routing_cost = value(model.alpha) * sum(
    value(model.dist[i, j]) * value(model.x[i, j, k])
    for (i, j) in model.ARCS for k in model.VEHICLES
)
service_penalty = value(model.beta) * sum(
    (value(model.b_final[i]) - value(model.t[i])) ** 2 for i in model.STATIONS
)
deployment_cost = value(model.gamma) * sum(
    value(model.x[i, j, k])
    for (i, j) in model.ARCS if i == "depot_start" and j != "depot_end"
    for k in model.VEHICLES
)
total_objective = value(model.obj)

# Extract time components
travel_time = sum(
    value(model.ttime[i, j]) * value(model.x[i, j, k])
    for (i, j) in model.ARCS for k in model.VEHICLES
)
station_time = sum(
    value(model.s) * sum(value(model.x[i_arc, j, k]) for (i_arc, j) in model.ARCS if i_arc == i for k in model.VEHICLES)
    + value(model.tau) * sum(value(model.u[i, k]) + value(model.v[i, k]) for k in model.VEHICLES)
    for i in model.STATIONS
)
depot_time = value(model.tau) * sum(
    value(model.u["depot_start", k]) + value(model.v["depot_end", k]) for k in model.VEHICLES
)
total_time = travel_time + station_time + depot_time

# Extract distance
total_distance_m = sum(
    value(model.dist[i, j]) * value(model.x[i, j, k])
    for (i, j) in model.ARCS for k in model.VEHICLES
)

# Extract MIP gap
try:
    obj_bound = float(result.problem.lower_bound)
    obj_value = float(result.problem.upper_bound)
    mip_gap = abs(obj_value - obj_bound) / abs(obj_value)
    mip_gap_pct = mip_gap * 100
except (AttributeError, TypeError, ZeroDivisionError):
    mip_gap_pct = None

# Count visited stations (fleet wide)
num_visited = sum(
    1 for h in model.STATIONS
    if sum(value(model.x[i, j, k]) for (i, j) in model.ARCS if j == h for k in model.VEHICLES) > 0.5
)

# Count vehicles used (leave the depot for a station)
num_vehicles_used = sum(
    1 for k in model.VEHICLES
    if sum(value(model.x[i, j, k]) for (i, j) in model.ARCS if i == "depot_start" and j != "depot_end") > 0.5
)

# Count total bikes moved (all pickups including initial depot load)
total_bikes_moved = sum(value(model.u[i, k]) for i in model.NODES for k in model.VEHICLES)

# Total squared deviation (independent of beta)
total_squared_deviation = sum(
    (value(model.b_final[i]) - value(model.t[i])) ** 2 for i in model.STATIONS
)

results_metrics = {
    "solver_status": str(result.solver.status),
    "termination_condition": str(result.solver.termination_condition),
    "objective_value": round(total_objective, 2),
    "mip_gap_percent": round(mip_gap_pct, 2) if mip_gap_pct is not None else None,
    "routing_cost": round(routing_cost, 1),
    "service_penalty": round(service_penalty, 1),
    "deployment_cost": round(deployment_cost, 1),
    "total_distance_m": round(total_distance_m, 1),
    "total_distance_km": round(total_distance_m / 1000, 3),
    "total_squared_deviation": round(total_squared_deviation, 1),
    "total_operation_time_min": round(total_time, 1),
    "travel_time_min": round(travel_time, 1),
    "station_time_min": round(station_time, 1),
    "depot_time_min": round(depot_time, 1),
    "travel_time_pct": round((travel_time / total_time * 100) if total_time > 0 else 0, 1),
    "station_time_pct": round((station_time / total_time * 100) if total_time > 0 else 0, 1),
    "depot_time_pct": round((depot_time / total_time * 100) if total_time > 0 else 0, 1),
    "total_bikes_moved": int(total_bikes_moved),
    "num_stations_visited": num_visited,
    "num_stations_available": len(stations),
    "num_vehicles_used": num_vehicles_used,
    "num_vehicles_available": K,
}

# with open(output_dir / "results_metrics.json", "w") as f:
#     json.dump(results_metrics, f, indent=2)

In [23]:
results_metrics

{'solver_status': 'ok',
 'termination_condition': 'optimal',
 'objective_value': 15327.43,
 'mip_gap_percent': 1.92,
 'routing_cost': 7872.4,
 'service_penalty': 7455.0,
 'total_distance_m': 7872.4,
 'total_distance_km': 7.872,
 'total_squared_deviation': 745.5,
 'total_operation_time_min': 178.8,
 'travel_time_min': 18.8,
 'station_time_min': 147.0,
 'depot_time_min': 13.0,
 'travel_time_pct': 10.5,
 'station_time_pct': 82.2,
 'depot_time_pct': 7.3,
 'total_bikes_moved': 45,
 'num_stations_visited': 14,
 'num_stations_available': 44}

In [ ]:
# ============================================================================
# 3. Results Stations CSV
# ============================================================================

ops_records = []
for i in model.NODES:
    pickups = int(round(sum(value(model.u[i, k]) for k in model.VEHICLES)))
    dropoffs = int(round(sum(value(model.v[i, k]) for k in model.VEHICLES)))

    if i in model.STATIONS:
        final = int(value(model.b_final[i]))
        target = value(model.t[i])
        initial_deviation = value(model.t[i]) - value(model.b[i])
        final_deviation = value(model.t[i]) - value(model.b_final[i])
    elif i == "depot_start":
        final = int(value(model.b[i])) - pickups
        target = None
        initial_deviation = None
        final_deviation = None
    else:  # depot_end
        final = int(value(model.b[i])) + dropoffs
        target = None
        initial_deviation = None
        final_deviation = None

    visited = sum(
        value(model.x[i_arc, j, k]) for (i_arc, j) in model.ARCS if i_arc == i for k in model.VEHICLES
    ) > 0.5

    ops_records.append(
        {
            "short_name": i,
            "capacity": int(value(model.c[i])),
            "target": target,
            "initial": int(value(model.b[i])),
            "pickups": pickups,
            "dropoffs": dropoffs,
            "final": final,
            "initial_deviation": initial_deviation,
            "final_deviation": final_deviation,
            "visited": int(visited),
        }
    )

results_stations = pd.DataFrame(ops_records)
# results_stations.to_csv(output_dir / "results_stations.csv", index=False)

In [25]:
results_stations

,short_name,capacity,target,initial,pickups,dropoffs,final,initial_deviation,final_deviation,visited
0,depot_start,20,NaN,20,5,0,15,NaN,NaN,1
1,A32003,11,-0.5,4,0,0,4,-4.5,-4.5,0
2,A32004,15,-2.5,3,0,0,3,-5.5,-5.5,0
3,A32008,15,8.5,14,2,0,12,-5.5,-3.5,1
4,A32056,25,1.5,1,0,1,2,0.5,-0.5,1
5,B32001,21,10.5,11,0,0,11,-0.5,-0.5,0
6,B32002,27,8.5,3,0,0,3,5.5,5.5,0
7,B32003,25,-4.5,1,0,1,2,-5.5,-6.5,1
8,B32005,39,21.5,34,9,0,25,-12.5,-3.5,1
9,B32006,15,1.5,14,9,0,5,-12.5,-3.5,1


In [ ]:
# ============================================================================
# 4. Route CSV
# ============================================================================

route_records = [
    {
        "vehicle": k,
        "from": i,
        "to": j,
        "pickups": int(value(model.u[i, k])),
        "dropoffs": int(value(model.v[i, k])),
        "load_leaving": int(value(model.w[i, k])),
    }
    for (i, j) in model.ARCS for k in model.VEHICLES
    if value(model.x[i, j, k]) > 0.5
]

route_df = pd.DataFrame(route_records)

# Order by sequence
route_df = order_route_by_sequence(route_df)

# route_df.to_csv(output_dir / "route.csv", index=False)

In [27]:
route_df

,from,to,pickups,dropoffs,load_leaving
0,depot_start,A32056,5,0,5
1,A32056,B32036,0,1,4
2,B32036,B32013,0,2,2
3,B32013,B32033,0,2,0
4,B32033,B32069,7,0,7
5,B32069,B32005,0,4,3
6,B32005,B32065,9,0,12
7,B32065,B32011,8,0,20
8,B32011,B32010,0,10,10
9,B32010,K32007,0,10,0


In [ ]:

# ============================================================================
# 5. Route Shapefile
# ============================================================================

# Merge route with network geometries
route_with_geom = route_df.merge(
    network_gdf[["origin", "dest", "geometry"]],
    left_on=["from", "to"],
    right_on=["origin", "dest"],
    how="left"
)

# Drop redundant columns and create GeoDataFrame
route_with_geom = route_with_geom.drop(columns=["origin", "dest"])
route_gdf = gpd.GeoDataFrame(route_with_geom, geometry="geometry", crs="EPSG:4326")

# # Save to shapefile
# route_shp_dir = output_dir / "route"
# route_shp_dir.mkdir(exist_ok=True)
# route_gdf.to_file(route_shp_dir / "route.shp")

# print(f"✓ All outputs saved to {output_dir}")

In [29]:
route_gdf

,from,to,pickups,dropoffs,load_leaving,geometry
0,depot_start,A32056,5,0,5,"LINESTRING (-71.10614 42.33828, -71.10714 42.3..."
1,A32056,B32036,0,1,4,"LINESTRING (-71.10979 42.33648, -71.10873 42.3..."
2,B32036,B32013,0,2,2,"LINESTRING (-71.10455 42.33401, -71.10423 42.3..."
3,B32013,B32033,0,2,0,"LINESTRING (-71.10465 42.33414, -71.10256 42.3..."
4,B32033,B32069,7,0,7,"LINESTRING (-71.09931 42.33616, -71.09931 42.3..."
5,B32069,B32005,0,4,3,"LINESTRING (-71.09931 42.33616, -71.09807 42.3..."
6,B32005,B32065,9,0,12,"LINESTRING (-71.08579 42.34337, -71.08684 42.3..."
7,B32065,B32011,8,0,20,"LINESTRING (-71.08774 42.3473, -71.08815 42.34..."
8,B32011,B32010,0,10,10,"LINESTRING (-71.09604 42.34908, -71.09674 42.3..."
9,B32010,K32007,0,10,0,"LINESTRING (-71.09716 42.34878, -71.09744 42.3..."
